# Spatial Analysis with Python
## A practical, hands-on tour

**Anzony Quispe** · 2026

Vector + raster + big geo data, all from a Jupyter Notebook.


## What you will leave with

- A mental model of **CRS**, **lat/lon**, points / lines / polygons
- The four operations you'll use 90% of the time:
  **intersection**, **overlay**, **dissolve / disaggregate**, **spatial join**
- How to read **Shapefile**, **GeoTIFF (.tif)**, **NetCDF (.nc)**, **HDF**, **GeoParquet**
- How to **rasterize / vectorize** and compute **zonal statistics**
- How to scale to big data with **Dask-GeoPandas** and **R-tree** indexes
- How to publish interactive maps with **folium**


## 0 · Setup

Install once (uncomment the line below if you are missing anything).


In [ ]:
# !pip install geopandas shapely pyproj rasterio rioxarray xarray netCDF4 h5py \
#     folium contextily mapclassify rasterstats pyarrow dask-geopandas pyogrio


In [ ]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["USE_PYGEOS"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPolygon, box

print("geopandas :", gpd.__version__)
print("shapely   :", __import__("shapely").__version__)


## 1 · Why Python for GIS?

| Click-based GIS (QGIS / ArcGIS) | Python (GeoPandas, rasterio, …) |
|---|---|
| Great for exploration | Great for **reproducibility** |
| One file at a time | Loops over thousands of files |
| Manual joins | Joins the entire pandas / numpy stack |
| Hard to scale | Dask, Parquet, parallel I/O |

> Rule of thumb: **explore in QGIS, ship in Python.**


## 2 · Latitude & Longitude

- **Latitude** (φ): −90° (South Pole) to +90° (North Pole)
- **Longitude** (λ): −180° to +180°, 0° at Greenwich
- A point on Earth = `(lon, lat)` — **x first, then y**
- These are **angles**, not distances. 1° of longitude ≠ 1° of latitude.

⚠️ In Shapely / GeoPandas the convention is `(x, y) = (lon, lat)`.
Mixing them up is the most common GIS bug.


In [ ]:
lima  = Point(-77.0428, -12.0464)   # (lon, lat)
cusco = Point(-71.9675, -13.5320)

print("Lima  :", lima,  " | x=lon", lima.x,  "y=lat", lima.y)
print("Cusco :", cusco)


## 3 · Coordinate Reference Systems (CRS)

A CRS tells software **how to interpret the numbers** in a geometry.

Two families:

| Family | Units | Example | Use for |
|---|---|---|---|
| **Geographic** | degrees | `EPSG:4326` (WGS84) | sharing data, web APIs |
| **Projected**  | meters  | `EPSG:3857` (Web Mercator), UTM zones | distances, areas, buffers |

EPSG codes are the universal IDs — look them up at https://epsg.io


### The three golden rules of CRS

1. **Every dataset has a CRS.** If it doesn't, you have a bug.
2. **Never compute distance/area in a geographic CRS.** Reproject first.
3. **All layers in an analysis must share the same CRS.**


In [ ]:
cities = gpd.GeoDataFrame(
    {"name": ["Lima", "Cusco", "Arequipa", "Iquitos"]},
    geometry=[
        Point(-77.0428, -12.0464),
        Point(-71.9675, -13.5320),
        Point(-71.5375, -16.3989),
        Point(-73.2536, -3.7437),
    ],
    crs="EPSG:4326",
)
cities


In [ ]:
print("CRS:", cities.crs)
print("Axis info:", cities.crs.axis_info[0].unit_name, "/", cities.crs.axis_info[1].unit_name)


## 4 · Distance — meters, not degrees

`.distance()` returns whatever unit the CRS uses.

- In `EPSG:4326` → degrees (almost never what you want)
- In a projected CRS → meters


In [ ]:
# WRONG: degrees
d_deg = cities.geometry.iloc[0].distance(cities.geometry.iloc[1])
print(f"Lima → Cusco in degrees (meaningless): {d_deg:.3f}")

# RIGHT: reproject to UTM 18S (Peru) and measure in meters
cities_m = cities.to_crs("EPSG:32718")
d_m = cities_m.geometry.iloc[0].distance(cities_m.geometry.iloc[1])
print(f"Lima → Cusco in km: {d_m/1000:,.1f}")


In [ ]:
# GeoPandas can pick a sensible UTM zone for you
auto = cities.estimate_utm_crs()
print("Auto-suggested CRS:", auto)
cities.to_crs(auto).geometry.distance(cities.to_crs(auto).geometry.shift()) / 1000


## 5 · The three geometric primitives

- **Point** — a city, a tree, a fire detection
- **LineString** — a road, a river segment, a flight path
- **Polygon** — a district, a lake, a building footprint
- And their *Multi-* variants for things that come in pieces (e.g., a country with islands)


In [ ]:
pt = Point(0, 0)
ln = LineString([(0, 0), (1, 1), (2, 0), (3, 2)])
pg = Polygon([(0, 0), (3, 0), (3, 2), (0, 2)], holes=[[(1, 0.5), (2, 0.5), (2, 1.5), (1, 1.5)]])

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
gpd.GeoSeries([pt]).plot(ax=axes[0], color="red", markersize=80); axes[0].set_title("Point")
gpd.GeoSeries([ln]).plot(ax=axes[1], linewidth=2);                axes[1].set_title("LineString")
gpd.GeoSeries([pg]).plot(ax=axes[2], color="lightgreen", edgecolor="k"); axes[2].set_title("Polygon w/ hole")
for a in axes: a.set_aspect("equal")
plt.tight_layout(); plt.show()


In [ ]:
# Geometries expose handy attributes
print("Area of pg :", pg.area)
print("Length of ln:", ln.length)
print("Bounds of pg:", pg.bounds)   # (minx, miny, maxx, maxy)
print("Is valid?   :", pg.is_valid)


## 6 · Centroids

The geometric center of a polygon (or line).

Used for:
- placing labels
- assigning a single coordinate to a region
- nearest-neighbor joins polygon → polygon

⚠️ Centroid of a polygon in `EPSG:4326` is **biased** — reproject first.


In [ ]:
peru_box = gpd.GeoDataFrame(
    {"name": ["Peru (bbox)"]},
    geometry=[box(-81.3, -18.4, -68.7, -0.0)],
    crs="EPSG:4326",
)

# bad: centroid in degrees
bad = peru_box.geometry.centroid.iloc[0]

# good: project to a metric CRS first
good = peru_box.to_crs(peru_box.estimate_utm_crs()).geometry.centroid.iloc[0]
good_ll = gpd.GeoSeries([good], crs=peru_box.estimate_utm_crs()).to_crs(4326).iloc[0]

print("Centroid (deg, biased) :", bad)
print("Centroid (UTM → back)  :", good_ll)


## 7 · Buffers

A buffer is **every point within distance _d_ of a geometry**.

Common uses:
- "everything within 500 m of a road"
- "1 km exclusion zone around a school"
- Convert points → polygons for joins

Buffer distance is in the **CRS units**, so reproject to meters first.


In [ ]:
cities_m = cities.to_crs(cities.estimate_utm_crs())

cities_m["buf_50km"] = cities_m.buffer(50_000)   # 50 km

ax = cities_m.set_geometry("buf_50km").plot(alpha=0.3, edgecolor="k")
cities_m.plot(ax=ax, color="red", markersize=40)
ax.set_title("50 km buffers around Peruvian cities (UTM 18S)")
plt.show()


## 8 · Reading & writing vector files

GeoPandas reads almost any vector format through `read_file`:

- **Shapefile** (`.shp` + sidecars) — old, ubiquitous
- **GeoPackage** (`.gpkg`) — modern, single file, recommended
- **GeoJSON** (`.geojson`) — text, web-friendly
- **GeoParquet** (`.parquet`) — columnar, huge files, lightning fast (we'll cover this later)


In [ ]:
# Download a small Natural Earth shapefile (1.1 MB) — countries of the world
import urllib.request, zipfile, io, tempfile, os

URL = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
tmp = tempfile.mkdtemp()
zip_path = os.path.join(tmp, "ne.zip")
urllib.request.urlretrieve(URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(tmp)

world = gpd.read_file(os.path.join(tmp, "ne_110m_admin_0_countries.shp"))
world = world[["NAME", "CONTINENT", "POP_EST", "geometry"]]
world.head()


In [ ]:
ax = world.plot(figsize=(10, 5), color="white", edgecolor="black", linewidth=0.3)
ax.set_title(f"World — {len(world)} countries, CRS = {world.crs.to_string()}")
plt.show()


In [ ]:
# Write to other formats (round-trip)
world.to_file("/tmp/world.gpkg", driver="GPKG")        # GeoPackage
world.to_file("/tmp/world.geojson", driver="GeoJSON")  # GeoJSON
print(os.listdir("/tmp/"))


## 9 · Intersection & overlay

Set-theoretic operations on geometries:

- **intersection** — the area in **both** A and B
- **union** — area in A **or** B
- **difference** — A **minus** B
- **symmetric difference** — XOR

`gpd.overlay(a, b, how=...)` applies these between two whole layers.


In [ ]:
south_america = world[world.CONTINENT == "South America"].copy()
equator_band = gpd.GeoDataFrame(
    {"name": ["±5° equator band"]},
    geometry=[box(-180, -5, 180, 5)],
    crs="EPSG:4326",
)

tropical_sa = gpd.overlay(south_america, equator_band, how="intersection")
ax = south_america.plot(color="lightgrey", edgecolor="white")
tropical_sa.plot(ax=ax, color="orange", edgecolor="k")
ax.set_title("South America ∩ equator band  (overlay how='intersection')")
plt.show()


In [ ]:
# difference: SA minus equator band
non_tropical = gpd.overlay(south_america, equator_band, how="difference")
non_tropical.plot(figsize=(6, 6), color="steelblue", edgecolor="k")
plt.title("South America − equator band")
plt.show()


## 10 · Dissolve & disaggregate

- **dissolve** → merge many polygons into one (group-by in space)
- **explode** → split a MultiPolygon back into its parts (disaggregate)

Together they let you move freely between "fine pieces" and "summary shapes".


In [ ]:
continents = world.dissolve(by="CONTINENT", aggfunc={"POP_EST": "sum"})
continents.plot(column="POP_EST", legend=True, figsize=(10, 5),
                cmap="YlOrRd", edgecolor="k")
plt.title("Dissolved by continent, colored by total population")
plt.show()
continents[["POP_EST"]].sort_values("POP_EST", ascending=False)


In [ ]:
# Disaggregate: each continent's MultiPolygon -> many Polygon rows
parts = continents.explode(index_parts=False).reset_index()
print(f"After explode: {len(parts)} pieces (was {len(continents)} continents)")
parts.head()


## 11 · Spatial joins — `sjoin` and `sjoin_nearest`

A regular join uses a key column. A **spatial join** uses geometry:

- `gpd.sjoin(left, right, predicate="within")` — which polygon does each point fall in?
- `predicate` can be `intersects`, `within`, `contains`, `touches`, `crosses`, ...
- `gpd.sjoin_nearest(left, right)` — nearest feature (e.g., nearest city to each fire)


In [ ]:
# Which country does each city belong to?
joined = gpd.sjoin(cities, world, predicate="within", how="left")
joined[["name", "NAME", "CONTINENT"]]


In [ ]:
# Nearest city to a set of random points in Peru
rng = np.random.default_rng(0)
random_pts = gpd.GeoDataFrame(
    {"id": range(5)},
    geometry=gpd.points_from_xy(
        rng.uniform(-78, -70, 5),
        rng.uniform(-15, -5, 5),
    ),
    crs="EPSG:4326",
)
nearest = gpd.sjoin_nearest(random_pts.to_crs(32718),
                            cities.to_crs(32718),
                            distance_col="dist_m")
nearest[["id", "name", "dist_m"]]


## 12 · Spatial indexing (R-tree)

Naïve "for every A check every B" is O(N·M).
A **spatial index** (R-tree) prunes pairs whose bounding boxes don't overlap → ~O(N log M).

GeoPandas builds one automatically under the hood (`gdf.sindex`).
You almost never call it directly, but it's why `sjoin` is fast even on millions of rows.


In [ ]:
big = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(
        np.random.uniform(-180, 180, 200_000),
        np.random.uniform(-60,  85,  200_000),
    ),
    crs="EPSG:4326",
)
print("Built an R-tree over", len(big), "points.")
print("Index type:", type(big.sindex).__name__)

# Query: which points fall inside South America's bounding box?
minx, miny, maxx, maxy = south_america.total_bounds
candidates = list(big.sindex.intersection((minx, miny, maxx, maxy)))
print("Candidates after R-tree prune:", len(candidates))


## 13 · Heavy data with **Apache GeoParquet**

Shapefiles are slow, capped at 2 GB, and have 10-char column names.
**GeoParquet** is the modern alternative:

- Columnar, compressed → 5-20× smaller than Shapefile
- Streams from cloud storage (S3, GCS) without a full download
- Stores CRS as metadata
- Native to the Arrow ecosystem (DuckDB, Polars, Dask)


In [ ]:
# Round-trip the world dataset to GeoParquet
world.to_parquet("/tmp/world.parquet")          # writes GeoParquet
back = gpd.read_parquet("/tmp/world.parquet")

import os
print("Shapefile size :", os.path.getsize("/tmp/world.gpkg"),    "bytes (gpkg)")
print("GeoJSON   size :", os.path.getsize("/tmp/world.geojson"), "bytes")
print("Parquet   size :", os.path.getsize("/tmp/world.parquet"), "bytes")
print("CRS preserved? ", back.crs == world.crs)


In [ ]:
# Filter on read — only read columns + rows you need
small = gpd.read_parquet(
    "/tmp/world.parquet",
    columns=["NAME", "CONTINENT", "geometry"],
)
small.head()


## 14 · Parallel processing with Dask-GeoPandas

When your GeoDataFrame won't fit in RAM (or is just slow):

- **dask-geopandas** partitions the data and runs ops in parallel
- Same API as GeoPandas (`.dissolve`, `.sjoin`, …)
- Reads/writes GeoParquet partitions directly


In [ ]:
import dask_geopandas as dgpd

# Pretend `big` is enormous; split into 8 partitions
dbig = dgpd.from_geopandas(big, npartitions=8)
print(dbig)

# Lazy: nothing has been computed yet
result = dbig.sjoin(south_america[["NAME", "geometry"]], predicate="within")
print("type:", type(result).__name__)

# Trigger computation
points_in_sa = result.compute()
print("Points in South America:", len(points_in_sa))


## 15 · Raster data — the other half of GIS

Rasters are **grids of pixels** (elevation, temperature, satellite imagery).

Key concepts:
- A raster has a **CRS** and an **affine transform** mapping (col, row) → (x, y)
- Each band is a 2-D array; multi-band rasters stack bands (e.g., RGB)

Main libraries:
- **rasterio** — low-level reads/writes of GeoTIFF
- **rioxarray** — labels rasterio arrays with `xarray` (recommended)
- **xarray** — N-dimensional labelled arrays, perfect for NetCDF


In [ ]:
import rasterio
from rasterio.transform import from_origin

# Build a synthetic 100x100 elevation raster centered on Lima
height, width = 100, 100
data = (np.indices((height, width)).sum(axis=0) * 10).astype("float32")
transform = from_origin(west=-77.2, north=-11.9, xsize=0.005, ysize=0.005)

with rasterio.open(
    "/tmp/elev.tif", "w",
    driver="GTiff", height=height, width=width, count=1,
    dtype=data.dtype, crs="EPSG:4326", transform=transform,
) as dst:
    dst.write(data, 1)

with rasterio.open("/tmp/elev.tif") as src:
    print("CRS      :", src.crs)
    print("Shape    :", src.shape)
    print("Bounds   :", src.bounds)
    print("Transform:", src.transform)


In [ ]:
import rioxarray as rxr
ras = rxr.open_rasterio("/tmp/elev.tif").squeeze()
print(ras)

ras.plot(cmap="terrain")
plt.title("Synthetic elevation raster (.tif)")
plt.show()


## 16 · NetCDF (`.nc`) — multi-dimensional climate data

NetCDF stores N-D arrays with labels (time × lat × lon × …).
Almost every climate / ocean / atmospheric dataset ships as NetCDF.

`xarray` is the tool of choice.


In [ ]:
import xarray as xr

# Build a synthetic temperature cube: (time, lat, lon)
times = pd.date_range("2024-01-01", periods=12, freq="MS")
lats = np.linspace(-20, 0, 21)
lons = np.linspace(-82, -68, 15)
temp = 25 - 0.5 * np.abs(lats[None, :, None]) \
         + 3 * np.sin(2 * np.pi * np.arange(12)[:, None, None] / 12) \
         + np.random.default_rng(1).normal(0, 0.5, (12, 21, 15))

ds = xr.Dataset(
    {"temperature": (("time", "lat", "lon"), temp.astype("float32"))},
    coords={"time": times, "lat": lats, "lon": lons},
    attrs={"description": "Synthetic monthly mean temperature"},
)
ds.to_netcdf("/tmp/temp.nc")
ds


In [ ]:
ds2 = xr.open_dataset("/tmp/temp.nc")
ds2["temperature"].mean("time").plot(cmap="coolwarm")
plt.title("Mean annual temperature (synthetic)")
plt.show()


## 17 · HDF (`.hdf`, `.h5`) — NASA / satellite data

HDF5 is the format behind MODIS, VIIRS, and many NASA products.
`h5py` reads any HDF5 file; `xarray` can open many HDF5 files directly.


In [ ]:
import h5py

# Write a tiny HDF5 file with two datasets
with h5py.File("/tmp/sample.h5", "w") as f:
    f.create_dataset("ndvi", data=np.random.rand(50, 50).astype("float32"))
    f.create_dataset("qa",   data=np.random.randint(0, 4, (50, 50), dtype="uint8"))
    f["ndvi"].attrs["units"] = "unitless"
    f.attrs["satellite"] = "FAKE-1"

with h5py.File("/tmp/sample.h5", "r") as f:
    print("Datasets:", list(f.keys()))
    print("ndvi shape :", f["ndvi"].shape, "units =", f["ndvi"].attrs["units"])
    print("satellite  :", f.attrs["satellite"])
    ndvi = f["ndvi"][:]

plt.imshow(ndvi, cmap="YlGn"); plt.colorbar(label="NDVI"); plt.title("HDF5 dataset"); plt.show()


## 18 · Rasterize ↔ Vectorize

- **Rasterize**: turn polygons into a pixel mask (e.g., country → raster)
- **Vectorize**: turn pixel regions into polygons (e.g., classified land cover → polygons)

Both live in `rasterio.features`.


In [ ]:
from rasterio import features
from rasterio.transform import from_bounds

# Rasterize South America at 1° resolution
minx, miny, maxx, maxy = south_america.total_bounds
res = 1.0
w = int((maxx - minx) / res); h = int((maxy - miny) / res)
trans = from_bounds(minx, miny, maxx, maxy, w, h)

mask = features.rasterize(
    [(g, 1) for g in south_america.geometry],
    out_shape=(h, w),
    transform=trans,
    fill=0,
    dtype="uint8",
)

plt.imshow(mask, extent=(minx, maxx, miny, maxy), origin="upper", cmap="Greens")
plt.title("South America rasterized at 1°"); plt.show()


In [ ]:
# Now go back the other way — vectorize the mask
shapes = list(features.shapes(mask, transform=trans))
polys = gpd.GeoDataFrame(
    {"value": [v for _, v in shapes]},
    geometry=[__import__("shapely").geometry.shape(g) for g, _ in shapes],
    crs="EPSG:4326",
)
polys = polys[polys.value == 1]
polys.plot(edgecolor="k", color="lightgreen")
plt.title(f"Vectorized back into {len(polys)} polygons")
plt.show()


## 19 · Zonal statistics

**Question**: for each polygon, what's the mean / max / sum of a raster inside it?

Examples:
- mean elevation per municipality
- total rainfall per watershed
- max NDVI per farm plot

Library: `rasterstats` (or do it manually with `rioxarray` + `xarray`).


In [ ]:
# Make 4 sample polygons covering the synthetic elevation raster
zones = gpd.GeoDataFrame(
    {"zone": ["NW", "NE", "SW", "SE"]},
    geometry=[
        box(-77.2, -12.15, -76.95, -11.9),
        box(-76.95, -12.15, -76.7, -11.9),
        box(-77.2, -12.4, -76.95, -12.15),
        box(-76.95, -12.4, -76.7, -12.15),
    ],
    crs="EPSG:4326",
)

from rasterstats import zonal_stats
stats = zonal_stats(zones, "/tmp/elev.tif", stats=["mean", "min", "max", "count"])
zones = zones.join(pd.DataFrame(stats))
zones[["zone", "mean", "min", "max", "count"]]


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ras.plot(ax=ax, cmap="terrain", alpha=0.6, add_colorbar=False)
zones.boundary.plot(ax=ax, color="red", linewidth=2)
for _, r in zones.iterrows():
    c = r.geometry.centroid
    ax.text(c.x, c.y, f"{r.zone}\n{r['mean']:.0f}", ha="center", va="center")
plt.title("Zonal mean elevation per zone")
plt.show()


## 20 · Interactive maps with **folium**

`matplotlib` is great for static plots. For a **web map with multiple layers and a layer switcher**, use folium (a Python wrapper around Leaflet.js).


In [ ]:
import folium
from folium.plugins import MarkerCluster

m = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="cartodbpositron")

# Layer 1: country polygon
folium.GeoJson(
    world[world.NAME == "Peru"].__geo_interface__,
    name="Peru border",
    style_function=lambda f: {"color": "#444", "weight": 2, "fillOpacity": 0.05},
).add_to(m)

# Layer 2: city points with popups
cluster = MarkerCluster(name="Cities").add_to(m)
for _, r in cities.iterrows():
    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        popup=r["name"],
        icon=folium.Icon(color="red", icon="info-sign"),
    ).add_to(cluster)

# Layer 3: 50 km buffer polygons (transformed back to lon/lat)
buf_ll = cities_m.set_geometry("buf_50km").to_crs(4326)
folium.GeoJson(
    buf_ll.__geo_interface__,
    name="50 km buffers",
    style_function=lambda f: {"color": "orange", "fillColor": "orange", "fillOpacity": 0.2},
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m


## 21 · Static basemaps with **contextily**

Add an OpenStreetMap / Carto / satellite background to a matplotlib plot in one line.
The trick: reproject your data to **Web Mercator (EPSG:3857)** first.


In [ ]:
import contextily as cx

ax = cities.to_crs(3857).plot(figsize=(8, 8), color="red", markersize=80)
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
ax.set_axis_off(); ax.set_title("Peruvian cities on OSM/Carto basemap")
plt.show()


## 22 · Cheatsheet

```python
import geopandas as gpd
gdf = gpd.read_file("file.shp")            # or .gpkg / .geojson / .parquet
gdf.crs                                     # check
gdf = gdf.to_crs(3857)                      # reproject (meters)
gdf["area_m2"]   = gdf.geometry.area
gdf["centroid"]  = gdf.geometry.centroid
gdf["buf_500m"]  = gdf.geometry.buffer(500)
joined = gpd.sjoin(points, polygons, predicate="within")
merged = polygons.dissolve(by="region", aggfunc="sum")
pieces = polygons.explode(index_parts=False)
inter  = gpd.overlay(a, b, how="intersection")
gdf.to_parquet("out.parquet")               # heavy data
```

```python
import rioxarray as rxr
r = rxr.open_rasterio("img.tif").squeeze()
r_clip = r.rio.clip(poly.geometry, poly.crs)
r.rio.to_raster("out.tif")
```


## 23 · Where to go next

- **Books**: *Geographic Data Science with Python* (Rey, Arribas-Bel, Wolf, 2023) — free online
- **Tutorials**: geopandas.org, automating-gis-processes.github.io
- **Big-data geo**: Dask-GeoPandas, Apache Sedona, DuckDB-spatial
- **Cloud-native rasters**: COG, STAC catalogs, Planetary Computer
- **Spatial statistics**: PySAL (Moran's I, hotspot analysis)
- **Networks / routing**: OSMnx + NetworkX

### Thank you!

Questions, ideas, corrections → anzony.quispe@gmail.com
